# 🎨 ComfyUI Studio - Google Colab Runtime

Notebook này cấu hình môi trường Google Colab thành GPU Runtime hiệu năng cao cho **ComfyUI Studio**:
- 💾 **Lưu trữ bền vững trên Google Drive** (`MyDrive/ComfyStudio/models`): tải một lần, lần sau mở lên dùng ngay, không mất thời gian tải lại.
- ⚡ **Tự động kiểm tra & bỏ qua (Skip)** nếu file model đã tồn tại trên Drive.
- 🖼️ **Tải sẵn bộ Model Qwen-Image-2.1** (INT8 Diffusion, INT8 Text Encoder Qwen3-VL 8B, BF16 VAE) có thanh tiến trình hiển thị % chi tiết.
- 🎛️ **Giao diện UI Dropdown tương tác**: Dễ dàng tải thêm Model, LoRA, VAE, ControlNet... vào đúng thư mục lưu trữ.
- 🌐 **Mở cổng Cloudflare Tunnel** kết nối trực tiếp đến ComfyUI Studio.



In [ ]:
#@title 1. Kết nối Google Drive & Khởi tạo Thư mục Models
import os
import shutil
from google.colab import drive

# 1. Mount Google Drive
print("🚀 Đang kết nối Google Drive...")
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio"
DRIVE_MODELS = os.path.join(DRIVE_BASE, "models")
DRIVE_OUTPUTS = os.path.join(DRIVE_BASE, "outputs")
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

# Các danh mục model chuẩn của ComfyUI
MODEL_CATEGORIES = [
    "checkpoints",
    "diffusion_models",
    "text_encoders",
    "clip",
    "loras",
    "vae",
    "controlnet",
    "upscale_models",
    "embeddings"
]

print(f"📁 Thư mục lưu ảnh tự động trên Google Drive: {DRIVE_OUTPUTS}")
print("📁 Khởi tạo cấu trúc thư mục lưu trữ models trên Google Drive...")
for cat in MODEL_CATEGORIES:
    path = os.path.join(DRIVE_MODELS, cat)
    os.makedirs(path, exist_ok=True)

# Tạo alias tương thích ngược (checkpoint <-> checkpoints, lora <-> loras, upscaler <-> upscale_models)
legacy_aliases = [
    ("checkpoint", "checkpoints"),
    ("lora", "loras"),
    ("upscaler", "upscale_models")
]
for old_name, new_name in legacy_aliases:
    old_p = os.path.join(DRIVE_MODELS, old_name)
    new_p = os.path.join(DRIVE_MODELS, new_name)
    if not os.path.exists(old_p) and os.path.exists(new_p):
        try:
            os.symlink(new_p, old_p)
        except Exception:
            pass

# Kiểm tra dung lượng Google Drive
try:
    total, used, free = shutil.disk_usage(DRIVE_BASE)
    print(f"✅ Google Drive đã kết nối thành công tại: {DRIVE_BASE}")
    print(f"📊 Dung lượng Drive: Đã dùng {used // (2**30)} GB / Tổng {total // (2**30)} GB (Còn trống {free // (2**30)} GB)")
except Exception as e:
    print(f"✅ Google Drive đã kết nối: {DRIVE_BASE}")


In [ ]:
#@title 2. Cài đặt ComfyUI & Cấu hình đường dẫn Models
import os
import subprocess
import time

COMFY_DIR = "/content/ComfyUI"

# 1. Clone ComfyUI với depth=1 để tối ưu tốc độ
if not os.path.exists(COMFY_DIR):
    print("🚀 Đang clone ComfyUI (phiên bản mới nhất)...")
    !git clone --depth=1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
else:
    print("ℹ️ ComfyUI đã có sẵn tại /content/ComfyUI.")

%cd /content/ComfyUI

# 2. Cài đặt PyTorch, requirements và các công cụ download
print("📦 Đang cài đặt PyTorch và thư viện phụ trợ...")
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu124
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q -r requirements.txt
!pip install -q huggingface_hub tqdm requests ipywidgets

# 3. Tạo cấu hình extra_model_paths.yaml để ComfyUI tự nhận diện toàn bộ models trong Google Drive
extra_paths_config = """
colab_drive:
    base_path: /content/drive/MyDrive/ComfyStudio/models
    checkpoints: checkpoints/
    diffusion_models: diffusion_models/
    unet: diffusion_models/
    text_encoders: text_encoders/
    clip: text_encoders/
    loras: loras/
    vae: vae/
    controlnet: controlnet/
    upscale_models: upscale_models/
    embeddings: embeddings/

colab_drive_legacy:
    base_path: /content/drive/MyDrive/ComfyStudio/models
    checkpoints: checkpoint/
    loras: lora/
    upscale_models: upscaler/
"""
with open('/content/ComfyUI/extra_model_paths.yaml', 'w') as f:
    f.write(extra_paths_config.strip())

# 4. Khởi tạo sẵn tất cả thư mục models bên trong ComfyUI
subdirs = [
    "checkpoints", "diffusion_models", "text_encoders", "clip",
    "loras", "vae", "controlnet", "upscale_models", "embeddings"
]
for s in subdirs:
    os.makedirs(f"/content/ComfyUI/models/{s}", exist_ok=True)

# 5. Cài đặt ComfyUI-Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    print("🔌 Đang cài đặt ComfyUI-Manager...")
    !git clone --depth=1 https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
else:
    print("ℹ️ ComfyUI-Manager đã sẵn sàng.")

print("\n✅ Cài đặt ComfyUI & cấu hình liên kết Google Drive thành công!")


In [ ]:
#@title 3. Tải Bộ Model Siêu Tốc (Lightning 2s, Turbo 1s, Realistic 5-20s)
#@markdown Tải trọn bộ model chuyên nghiệp cho tạo ảnh siêu nhanh và chất lượng cao:
#@markdown - ⚡⚡ **SDXL Lightning 4-Step** (ByteDance): `checkpoints/sdxl_lightning_4step.safetensors` (~6.5 GB) - **Gen 1 ảnh chỉ 2 giây!**
#@markdown - ⚡⚡ **SDXL Turbo** (Stability AI): `checkpoints/sd_xl_turbo_1.0_fp16.safetensors` (~3.3 GB) - **Gen 1 ảnh chỉ 1 giây!** (512x512)
#@markdown - ⚡ **RealVisXL V4.0 (SDXL)**: `checkpoints/realvisxlV40_v40Bakedvae.safetensors` (~6.6 GB) - Chất lượng cao nhất 15-20s.
#@markdown - ⚡ **Realistic Vision V6.0 (SD1.5)**: `checkpoints/realisticVisionV60B1_v51VAE.safetensors` (~2.1 GB) - 5-8s.
#@markdown - 🎛️ **SDXL VAE Fix** + **4x-UltraSharp** + **LoRA Add-Detail-XL** + **ControlNet OpenPose**
#@markdown
#@markdown *Chỉ tải 1 lần vào Google Drive. Các lần sau tự bỏ qua.*

import os
import sys
import inspect
from huggingface_hub import hf_hub_download

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
COMFY_DIR = "/content/ComfyUI/models"
has_drive = os.path.exists(DRIVE_DIR)
TARGET_BASE = DRIVE_DIR if has_drive else COMFY_DIR

FAST_REALISTIC_MODELS = [
    {
        "desc": "1. SDXL Lightning 4-Step (ByteDance) - Gen 2 giây!",
        "repo_id": "ByteDance/SDXL-Lightning",
        "category": "checkpoints",
        "rel_path": "sdxl_lightning_4step.safetensors",
        "basename": "sdxl_lightning_4step.safetensors",
        "min_size_mb": 5000
    },
    {
        "desc": "2. SDXL Turbo FP16 (Stability AI) - Gen 1 giây!",
        "repo_id": "stabilityai/sdxl-turbo",
        "category": "checkpoints",
        "rel_path": "sd_xl_turbo_1.0_fp16.safetensors",
        "basename": "sd_xl_turbo_1.0_fp16.safetensors",
        "min_size_mb": 3000
    },
    {
        "desc": "3. SDXL Realistic Checkpoint: RealVisXL V4.0 (Render 15-20s)",
        "repo_id": "SG161222/RealVisXL_V4.0",
        "category": "checkpoints",
        "rel_path": "RealVisXL_V4.0.safetensors",
        "basename": "realvisxlV40_v40Bakedvae.safetensors",
        "min_size_mb": 5000
    },
    {
        "desc": "4. SD1.5 Checkpoint: Realistic Vision V5.1/V6.0 (Render 5s)",
        "repo_id": "SG161222/Realistic_Vision_V5.1_noVAE",
        "category": "checkpoints",
        "rel_path": "Realistic_Vision_V5.1_fp16-no-ema.safetensors",
        "basename": "realisticVisionV60B1_v51VAE.safetensors",
        "min_size_mb": 1800
    },
    {
        "desc": "5. SDXL VAE (Khử mờ & cân chỉnh màu)",
        "repo_id": "stabilityai/sdxl-vae",
        "category": "vae",
        "rel_path": "sdxl_vae.safetensors",
        "basename": "sdxl_vae.safetensors",
        "min_size_mb": 200
    },
    {
        "desc": "6. AI Upscaler 4x-UltraSharp (Nâng nét ảnh 4k)",
        "repo_id": "lokcx/4x-Ultrasharp",
        "category": "upscale_models",
        "rel_path": "4x-UltraSharp.pth",
        "basename": "4x-UltraSharp.pth",
        "min_size_mb": 50
    },
    {
        "desc": "7. LoRA Add-Detail-XL (Tăng chi tiết da, mắt, tóc)",
        "repo_id": "OedoSoldier/detail-tweaker-lora",
        "category": "loras",
        "rel_path": "add-detail-xl.safetensors",
        "basename": "add-detail-xl.safetensors",
        "min_size_mb": 30
    },
    {
        "desc": "8. ControlNet OpenPose (Kiểm soát dáng người mẫu)",
        "repo_id": "lllyasviel/ControlNet-v1-1",
        "category": "controlnet",
        "rel_path": "control_v11p_sd15_openpose.pth",
        "basename": "control_v11p_sd15_openpose.pth",
        "min_size_mb": 1000
    }
]

print(f"🎯 Vị trí lưu trữ: {TARGET_BASE} (Google Drive bền vững)")

for item in FAST_REALISTIC_MODELS:
    desc = item["desc"]
    repo_id = item["repo_id"]
    category = item["category"]
    rel_path = item["rel_path"]
    basename = item["basename"]
    min_bytes = item["min_size_mb"] * 1024 * 1024

    drive_file = os.path.join(DRIVE_DIR, category, basename) if has_drive else None
    comfy_file = os.path.join(COMFY_DIR, category, basename)

    # 1. Kiểm tra nếu đã có trên Google Drive
    if drive_file and os.path.exists(drive_file) and os.path.getsize(drive_file) >= min_bytes:
        size_gb = os.path.getsize(drive_file) / (1024 ** 3)
        print(f"⏭️  [ĐÃ CÓ TRÊN DRIVE] {desc} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
        if not os.path.exists(comfy_file):
            try:
                os.makedirs(os.path.dirname(comfy_file), exist_ok=True)
                os.symlink(drive_file, comfy_file)
                print(f"    🔗 Đã liên kết symlink sang ComfyUI: {comfy_file}")
            except Exception:
                pass
        continue

    # 2. Kiểm tra nếu đã có trên local ComfyUI
    if os.path.exists(comfy_file) and os.path.getsize(comfy_file) >= min_bytes:
        size_gb = os.path.getsize(comfy_file) / (1024 ** 3)
        print(f"⏭️  [ĐÃ CÓ TRÊN COMFYUI] {desc} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
        continue

    # 3. Tải từ HuggingFace
    print(f"\n⬇️  Đang tải {desc}...")
    print(f"   Repo: {repo_id}/{rel_path} -> {category}/{basename}")

    dest_dir = os.path.join(TARGET_BASE, category)
    os.makedirs(dest_dir, exist_ok=True)

    try:
        download_kwargs = {
            "repo_id": repo_id,
            "filename": rel_path,
            "local_dir": dest_dir
        }
        sig = inspect.signature(hf_hub_download).parameters
        if "local_dir_use_symlinks" in sig:
            download_kwargs["local_dir_use_symlinks"] = False

        downloaded_path = hf_hub_download(**download_kwargs)
        final_target = os.path.join(dest_dir, basename)
        if downloaded_path != final_target and os.path.exists(downloaded_path):
            if os.path.exists(final_target):
                os.remove(final_target)
            os.rename(downloaded_path, final_target)

        print(f"✅ Đã tải xong: {desc}")

        if has_drive and os.path.exists(drive_file) and not os.path.exists(comfy_file):
            try:
                os.makedirs(os.path.dirname(comfy_file), exist_ok=True)
                os.symlink(drive_file, comfy_file)
                print(f"   🔗 Đã liên kết symlink sang ComfyUI: {comfy_file}")
            except Exception:
                pass
    except Exception as e:
        print(f"⚠️ Không tải được {basename}: {e}")

print("\n🎉 Hoàn thành kiểm tra & tải bộ Model Siêu Tốc (SDXL, SD1.5, VAE, LoRA, Upscaler)!")


In [ ]:
#@title 3B. Tải bộ Model Qwen-Image-2.1 (Tùy chọn - Dành cho ai muốn dùng Qwen)
#@markdown Tải bộ 3 model Qwen-Image-2.1 phục vụ workflow Text-to-Image (Lưu ý: Model nặng 30GB, gen ~20p):
#@markdown - 1. **Diffusion Model (INT8)**: `diffusion_models/qwen_image_2.1_int8_convrot.safetensors` (~12.3 GB)
#@markdown - 2. **Text Encoder Qwen3-VL 8B (INT8)**: `text_encoders/qwen3vl_8b_int8_convrot.safetensors` (~8.8 GB)
#@markdown - 3. **VAE (BF16)**: `vae/qwen_image_2.1_vae_bf16.safetensors` (~254 MB)
#@markdown
#@markdown *Tiến trình hiển thị % dung lượng tải thực tế. Nếu file đã có trên Drive sẽ tự động bỏ qua.*

import os
import sys
import inspect
from huggingface_hub import hf_hub_download

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
COMFY_DIR = "/content/ComfyUI/models"
has_drive = os.path.exists(DRIVE_DIR)

# Danh sách 3 model cần tải theo cấu trúc chuẩn ComfyUI
QWEN_MODELS = [
    {
        "desc": "1. Diffusion Model (INT8)",
        "repo_id": "Comfy-Org/Qwen-Image-2.1",
        "category": "diffusion_models",
        "rel_path": "diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        "basename": "qwen_image_2.1_int8_convrot.safetensors",
        "min_size_mb": 5000 # tối thiểu ~5GB
    },
    {
        "desc": "2. Text Encoder Qwen3-VL 8B (INT8)",
        "repo_id": "Comfy-Org/Qwen-Image-2.1",
        "category": "text_encoders",
        "rel_path": "text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        "basename": "qwen3vl_8b_int8_convrot.safetensors",
        "min_size_mb": 4000 # tối thiểu ~4GB
    },
    {
        "desc": "3. VAE (BF16)",
        "repo_id": "Comfy-Org/Qwen-Image-2.1",
        "category": "vae",
        "rel_path": "vae/qwen_image_2.1_vae_bf16.safetensors",
        "basename": "qwen_image_2.1_vae_bf16.safetensors",
        "min_size_mb": 100 # tối thiểu ~100MB
    }
]

# Thư mục gốc lưu trữ: ưu tiên Google Drive để lần sau không phải tải lại
TARGET_BASE = DRIVE_DIR if has_drive else COMFY_DIR
print(f"🎯 Vị trí lưu trữ mục tiêu: {TARGET_BASE}")
if has_drive:
    print("💡 Đã phát hiện Google Drive. Model sẽ được lưu vào Drive để dùng vĩnh viễn!")
else:
    print("⚠️ Chưa mount Google Drive! Model sẽ được lưu tạm thời trên Colab.")

for item in QWEN_MODELS:
    desc = item["desc"]
    repo_id = item["repo_id"]
    category = item["category"]
    rel_path = item["rel_path"]
    basename = item["basename"]
    min_bytes = item["min_size_mb"] * 1024 * 1024

    drive_file = os.path.join(DRIVE_DIR, category, basename) if has_drive else None
    comfy_file = os.path.join(COMFY_DIR, category, basename)

    # 1. Kiểm tra nếu đã có trên Google Drive
    if drive_file and os.path.exists(drive_file) and os.path.getsize(drive_file) >= min_bytes:
        size_gb = os.path.getsize(drive_file) / (1024 ** 3)
        print(f"⏭️  [ĐÃ CÓ TRÊN DRIVE] {desc} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
        # Đảm bảo ComfyUI có symlink đến file trong Drive
        if not os.path.exists(comfy_file):
            try:
                os.makedirs(os.path.dirname(comfy_file), exist_ok=True)
                os.symlink(drive_file, comfy_file)
                print(f"    🔗 Đã liên kết symlink sang ComfyUI: {comfy_file}")
            except Exception:
                pass
        continue

    # 2. Kiểm tra nếu đã có trên local ComfyUI
    if os.path.exists(comfy_file) and os.path.getsize(comfy_file) >= min_bytes:
        size_gb = os.path.getsize(comfy_file) / (1024 ** 3)
        print(f"⏭️  [ĐÃ CÓ TRÊN COMFYUI] {desc} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
        continue

    # 3. Tải model với tiến trình % (tqdm/huggingface_hub)
    print(f"\n⬇️  Đang tải {desc}: {basename}...")
    print(f"   Repo: {repo_id} | Thư mục đích: {category}/")

    # Xác định tham số tương thích cho hf_hub_download
    download_kwargs = {
        "repo_id": repo_id,
        "filename": rel_path,
        "local_dir": TARGET_BASE
    }
    sig = inspect.signature(hf_hub_download).parameters
    if "local_dir_use_symlinks" in sig:
        download_kwargs["local_dir_use_symlinks"] = False

    # Thực hiện tải (có hiển thị % tiến trình, dung lượng và tốc độ MB/s qua tqdm)
    downloaded_path = hf_hub_download(**download_kwargs)
    print(f"✅ Đã tải xong: {desc}")

    # Nếu tải vào Drive, tạo symlink sang ComfyUI để ComfyUI thấy ngay lập tức
    if has_drive and os.path.exists(drive_file) and not os.path.exists(comfy_file):
        try:
            os.makedirs(os.path.dirname(comfy_file), exist_ok=True)
            os.symlink(drive_file, comfy_file)
            print(f"   🔗 Đã liên kết symlink sang ComfyUI: {comfy_file}")
        except Exception:
            pass

print("\n🎉 Hoàn thành kiểm tra & tải trọn bộ model Qwen-Image-2.1!")


In [ ]:
#@title 🧨 4. Download Models (Tải Checkpoint, LoRA, VAE, ControlNet - Tương thích SDVN)
#@markdown Hỗ trợ link Civitai (civitai.com, civitai.red), HuggingFace, Direct URL.
#@markdown - ⚡ **Bypass 403 Forbidden**: Tự động nhận diện `modelVersionId` và tải qua API token.
#@markdown - ⚡ **Tốc độ cao**: Sử dụng `aria2c` 16 kết nối song song, tự động tiếp tục tải nếu đứt mạng.
#@markdown - ⚡ **Cú pháp đổi tên**: Thêm `@= ten_file.safetensors` vào sau link nếu muốn đổi tên.
#@markdown - ⚡ **Bảo vệ Google Drive**: Tự động kiểm tra file đã có để bỏ qua, tự động tạo Symlink vào ComfyUI.

#@markdown ### 📥 Nhập link theo danh mục (hỗ trợ nhiều link cách nhau bằng dấu phẩy hoặc xuống dòng):
Checkpoint_Links = "" #@param {type:"string"}
Lora_Links = "" #@param {type:"string"}
Controlnet_Links = "" #@param {type:"string"}
Vae_Links = "" #@param {type:"string"}
Diffusion_Model_Links = "" #@param {type:"string"}
Text_Encoder_Links = "" #@param {type:"string"}
Upscale_Model_Links = "" #@param {type:"string"}

#@markdown ---
#@markdown ### ✨ Tùy chọn Custom Download & API Tokens:
Custom_Category = "checkpoints" #@param ["checkpoints", "diffusion_models", "loras", "text_encoders", "vae", "controlnet", "upscale_models", "embeddings"]
Custom_Link = "" #@param {type:"string"}
Custom_File_Name = "" #@param {type:"string"}
Quick_Preset = "-- Kh\u00f4ng d\u00f9ng preset (D\u00f9ng link \u1edf tr\u00ean) --" #@param ["-- Kh\u00f4ng d\u00f9ng preset (D\u00f9ng link \u1edf tr\u00ean) --", "[Krea 2] LUSTIFY Turbo int8 (12 GB)", "[LoRA] Krea 2 NSFW V4 LoRA (435 MB)", "[Krea 2] Text Encoder Qwen3-VL 4B (4.4 GB)", "[Krea 2] VAE Wan 2.1 (254 MB)", "[Checkpoint] Realistic Vision V6.0 Hyper (2 GB)", "[SDXL Checkpoint] SDXL Base 1.0 (6.6 GB)", "[SDXL Checkpoint] Juggernaut XL v9 (6.6 GB)", "[Upscaler] 4x-UltraSharp (67 MB)", "[VAE] SDXL VAE (335 MB)"]
Civitai_API_Token = "8c7337ac0c39fe4133ae19a3d65b806f" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

import os
import re
import sys
import json
import subprocess
import urllib.request

# 1. Đảm bảo aria2c đã sẵn sàng
!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

# 2. Cấu trúc thư mục tương thích ComfyStudio
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
COMFY_BASE = "/content/ComfyUI/models"
has_drive = os.path.exists("/content/drive/MyDrive")
TARGET_BASE = DRIVE_BASE if has_drive else COMFY_BASE

print(f"🎯 Thư mục lưu trữ chính: {TARGET_BASE}")
if has_drive:
    print("💾 Google Drive đã kết nối -> Model sẽ được lưu vĩnh viễn trên Drive!")
else:
    print("⚠️ Google Drive chưa mount -> Model lưu tạm trên ổ cứng Colab.")

def split_name(link):
    delimiter = "@="
    if delimiter in link:
        parts = link.split(delimiter, 1)
        url = parts[0].strip()
        name = parts[1].strip()
        valid_exts = ('.safetensors', '.ckpt', '.pt', '.pth', '.bin', '.gguf', '.onnx')
        if not any(name.lower().endswith(ext) for ext in valid_exts):
            name = f"{name}.safetensors"
        return url, name
    return link.strip(), None

def parse_links(raw_text):
    if not raw_text:
        return []
    items = []
    for line in raw_text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        for part in line.split(','):
            part = part.strip()
            if part:
                items.append(part)
    return items

def resolve_url_and_filename(url, civitai_token="", hf_token=""):
    url = url.strip()
    filename = None
    
    # A. HuggingFace: Đổi /blob/ thành /resolve/
    if "huggingface.co" in url:
        if "/blob/" in url:
            url = url.replace("/blob/", "/resolve/")
        if hf_token and "token=" not in url:
            sep = "&" if "?" in url else "?"
            url = f"{url}{sep}token={hf_token}"
        clean_path = url.split("?")[0]
        filename = clean_path.split("/")[-1] if "/" in clean_path else None
        return url, filename
        
    # B. CivitAI: Tự động trích xuất modelVersionId hoặc Model ID
    if "civitai.com" in url or "civitai.red" in url:
        m_ver = re.search(r'modelVersionId=(\d+)', url)
        vid = m_ver.group(1) if m_ver else None
        
        if not vid:
            m_api = re.search(r'/(?:api/)?download/models/(\d+)', url)
            if m_api:
                vid = m_api.group(1)
                
        # Nếu có modelVersionId -> Lấy metadata tên file từ CivitAI API
        if vid:
            try:
                req = urllib.request.Request(f"https://civitai.com/api/v1/model-versions/{vid}", headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=6) as r:
                    d = json.loads(r.read().decode())
                    files = d.get('files', [])
                    if files and 'name' in files[0]:
                        filename = files[0]['name']
            except Exception:
                pass
            dl_url = f"https://civitai.com/api/download/models/{vid}"
            if civitai_token:
                dl_url += f"?token={civitai_token}"
            return dl_url, filename

        # Nếu chỉ có Model ID chung (/models/12345)
        m_model = re.search(r'/models/(\d+)', url)
        if m_model:
            model_id = m_model.group(1)
            try:
                req = urllib.request.Request(f"https://civitai.com/api/v1/models/{model_id}", headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=6) as r:
                    d = json.loads(r.read().decode())
                    versions = d.get('modelVersions', [])
                    if versions:
                        vid = versions[0].get('id')
                        files = versions[0].get('files', [])
                        filename = files[0].get('name') if files else None
                        dl_url = f"https://civitai.com/api/download/models/{vid}"
                        if civitai_token:
                            dl_url += f"?token={civitai_token}"
                        return dl_url, filename
            except Exception:
                pass
                
        if civitai_token and "token=" not in url:
            sep = "&" if "?" in url else "?"
            url = f"{url}{sep}token={civitai_token}"
        return url, filename

    # C. Direct URL bình thường
    clean_path = url.split("?")[0]
    filename = clean_path.split("/")[-1] if "/" in clean_path else None
    return url, filename

def sync_symlinks(category):
    if not has_drive:
        return
    drive_cat_dir = os.path.join(DRIVE_BASE, category)
    comfy_cat_dir = os.path.join(COMFY_BASE, category)
    os.makedirs(comfy_cat_dir, exist_ok=True)
    if os.path.exists(drive_cat_dir):
        for f in os.listdir(drive_cat_dir):
            src = os.path.join(drive_cat_dir, f)
            dst = os.path.join(comfy_cat_dir, f)
            if os.path.isfile(src) and not os.path.exists(dst):
                try:
                    os.symlink(src, dst)
                    print(f"   🔗 Symlink sang ComfyUI: {category}/{f}")
                except Exception:
                    pass

# 3. Khởi tạo danh sách tải
PRESET_MAP = {
    "[Krea 2] LUSTIFY Turbo int8 (12 GB)": ("diffusion_models", "https://civitai.com/api/download/models/3112728?fileId=2997637"),
    "[LoRA] Krea 2 NSFW V4 LoRA (435 MB)": ("loras", "https://civitai.com/api/download/models/3147117?fileId=3027612"),
    "[Krea 2] Text Encoder Qwen3-VL 4B (4.4 GB)": ("text_encoders", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"),
    "[Krea 2] VAE Wan 2.1 (254 MB)": ("vae", "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/wan_2.1_vae.safetensors"),
    "[Checkpoint] Realistic Vision V6.0 Hyper (2 GB)": ("checkpoints", "https://civitai.com/api/download/models/501240?fileId=418901"),
    "[SDXL Checkpoint] SDXL Base 1.0 (6.6 GB)": ("checkpoints", "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"),
    "[SDXL Checkpoint] Juggernaut XL v9 (6.6 GB)": ("checkpoints", "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"),
    "[Upscaler] 4x-UltraSharp (67 MB)": ("upscale_models", "https://huggingface.co/lokcx/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth"),
    "[VAE] SDXL VAE (335 MB)": ("vae", "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors"),
}

queue = []
category_inputs = [
    ("checkpoints", Checkpoint_Links),
    ("loras", Lora_Links),
    ("controlnet", Controlnet_Links),
    ("vae", Vae_Links),
    ("diffusion_models", Diffusion_Model_Links),
    ("text_encoders", Text_Encoder_Links),
    ("upscale_models", Upscale_Model_Links),
]

for cat, links_text in category_inputs:
    for item in parse_links(links_text):
        url, name = split_name(item)
        queue.append((cat, url, name))

if Custom_Link.strip():
    for item in parse_links(Custom_Link):
        url, name = split_name(item)
        if Custom_File_Name.strip():
            name = Custom_File_Name.strip()
        queue.append((Custom_Category, url, name))

if Quick_Preset in PRESET_MAP:
    cat, p_url = PRESET_MAP[Quick_Preset]
    queue.append((cat, p_url, None))

# 4. Tiến hành tải từng file
if not queue:
    print("ℹ️ Chưa có link nào được nhập. Hãy dán link vào các ô ở trên rồi nhấn Chạy ô này!")
else:
    print(f"\n📋 Tổng số file cần tải: {len(queue)}")
    for idx, (cat, raw_url, explicit_name) in enumerate(queue, 1):
        dest_dir = os.path.join(TARGET_BASE, cat)
        comfy_dir = os.path.join(COMFY_BASE, cat)
        os.makedirs(dest_dir, exist_ok=True)
        os.makedirs(comfy_dir, exist_ok=True)
        
        print(f"\n------------------------------------------------------------")
        print(f"[{idx}/{len(queue)}] Xử lý link: {raw_url}")
        
        # Phân tích URL và xác định tên file
        download_url, auto_name = resolve_url_and_filename(raw_url, Civitai_API_Token, HuggingFace_Token)
        final_name = explicit_name or auto_name
        
        # Kiểm tra nếu file đã tồn tại trên Drive hoặc ComfyUI
        if final_name:
            target_path = os.path.join(dest_dir, final_name)
            comfy_path = os.path.join(comfy_dir, final_name)
            if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
                size_gb = os.path.getsize(target_path) / (1024 ** 3)
                print(f"⏭️  [ĐÃ CÓ SẴN] {cat}/{final_name} ({size_gb:.2f} GB) -> BỎ QUA TẢI LẠI.")
                if has_drive and not os.path.exists(comfy_path):
                    try:
                        os.symlink(target_path, comfy_path)
                        print(f"    🔗 Symlink sang ComfyUI: {comfy_path}")
                    except Exception:
                        pass
                continue
        
        print(f"⬇️  Đích đến: {cat}/ ({final_name if final_name else 'Tự lấy tên từ server'})")
        print(f"🌐 Link tải trực tiếp: {download_url}")
        
        # Sử dụng aria2c đa luồng (16 kết nối)
        if final_name:
            cmd = f'aria2c --console-log-level=warn --summary-interval=5 -c -x 16 -s 16 -k 1M --check-certificate=false "{download_url}" -d "{dest_dir}" -o "{final_name}"'
        else:
            cmd = f'aria2c --console-log-level=warn --summary-interval=5 -c -x 16 -s 16 -k 1M --check-certificate=false --content-disposition=true "{download_url}" -d "{dest_dir}"'
            
        res = subprocess.run(cmd, shell=True)
        
        # Nếu aria2c gặp lỗi, thử lại bằng wget dự phòng
        if res.returncode != 0:
            print("⚠️ aria2c gặp sự cố, thử lại bằng wget...")
            if final_name:
                wget_cmd = f'wget -c "{download_url}" -O "{dest_dir}/{final_name}"'
            else:
                wget_cmd = f'wget -c --content-disposition "{download_url}" -P "{dest_dir}"'
            subprocess.run(wget_cmd, shell=True)
            
        # Đồng bộ symlink sang ComfyUI
        sync_symlinks(cat)
        print(f"✅ Đã tải xong vào mục {cat}!")

    print("\n🎉 Hoàn thành toàn bộ tác vụ tải model!")


In [ ]:
#@title 5. Khởi chạy ComfyUI & Mở Tunnel (Ngrok / LocalTunnel)
#@markdown Cloudflare thường xuyên chặn ComfyUI (Lỗi 403 Forbidden). Hãy dùng **LocalTunnel** (miễn phí, không cần tài khoản) hoặc **Ngrok** (nhanh, ổn định, cần authtoken).

tunnel_type = "localtunnel (Miễn phí, không cần Token)" #@param ["localtunnel (Miễn phí, không cần Token)", "ngrok (Cần Auth Token)"]
ngrok_auth_token = "" #@param {type:"string"}

import subprocess
import time
import re
import os

# 1. Khởi chạy ComfyUI dưới nền
print("🚀 Đang khởi chạy ComfyUI backend (Output tự động lưu vào Google Drive)...")
log_file = open("/content/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    [
        "python", "/content/ComfyUI/main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--preview-method", "auto",
        "--output-directory", "/content/drive/MyDrive/ComfyStudio/outputs"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# 2. Mở Tunnel
if "ngrok" in tunnel_type.lower():
    if not ngrok_auth_token:
        print("❌ BẠN CHƯA NHẬP NGROK AUTH TOKEN!")
        print("Hãy lấy token tại https://dashboard.ngrok.com/get-started/your-authtoken")
    else:
        print("🔗 Đang thiết lập Ngrok Tunnel...")
        !pip install pyngrok -q
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = ngrok_auth_token
        public_url = ngrok.connect(8188).public_url
        print("\n=======================================================")
        print(f"🎉 COMFYUI STUDIO ENDPOINT: {public_url}")
        print("👉 Sao chép URL trên và dán vào ComfyUI Studio -> Settings / Runtime")
        print("=======================================================\n")
else:
    print("🔗 Đang thiết lập LocalTunnel...")
    !npm install -g localtunnel > /dev/null 2>&1
    tunnel_proc = subprocess.Popen(
        ['lt', '--port', '8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in tunnel_proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.loca\.lt', line)
        if match:
            url = match.group(0)
            print("\n=======================================================")
            print(f"🎉 COMFYUI STUDIO ENDPOINT: {url}")
            print("👉 Sao chép URL trên và dán vào ComfyUI Studio -> Settings / Runtime")
            print("=======================================================\n")
            break
